<a href="https://colab.research.google.com/github/RafaelaMlucca/AnaliseViolMulher/blob/main/notebooks/01_ingestao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ingestão de Dados — SINAN

**Projeto:** Assinaturas preditivas dos tipos de violência contra a mulher  
**Autora:** Rafaela Lucca

Aquisição dos microdados do SINAN-VIOL (2018-2024) via `pysus`, filtrando notificações de mulheres e selecionando as variáveis da análise.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pysus --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 753.0 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import gc
import glob
from pathlib import Path
import pandas as pd

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')
PARTES = DRIVE / 'partes'
DRIVE.mkdir(parents=True, exist_ok=True)
PARTES.mkdir(parents=True, exist_ok=True)

print(f'Pasta de trabalho: {DRIVE}')

Pasta de trabalho: /content/drive/MyDrive/projeto_violencia_mulher


## Lista de variáveis selecionadas

Total: **55 variáveis** organizadas em 9 grupos.

In [4]:
# === IDENTIFICAÇÃO TEMPORAL E GEOGRÁFICA ===
TEMPORAL = [
    'NU_ANO',        # ano da notificação (chave para split temporal)
    'DT_NOTIFIC',    # data da notificação
    'DT_OCOR',       # data da ocorrência
    'ANO_NASC',      # ano de nascimento da vítima
]

GEOGRAFICA = [
    'SG_UF_OCOR',    # UF onde a violência ocorreu
    'ID_MN_OCOR',    # município de ocorrência
    'ID_MN_RESI',    # município de residência (mobilidade vítima/local)
]

# === PERFIL DA VÍTIMA ===
VITIMA_PERFIL = [
    'CS_SEXO',       # filtro: F (será descartado depois)
    'NU_IDADE_N',    # idade codificada (será reconstruída)
    'CS_GESTANT',    # situação gestacional
    'CS_RACA',       # raça/cor (autodeclarada)
    'CS_ESCOL_N',    # escolaridade
    'SIT_CONJUG',    # situação conjugal
    'ORIENT_SEX',    # orientação sexual
    'IDENT_GEN',     # identidade de gênero
    'CICL_VID',      # ciclo de vida (criança/adolescente/jovem/adulta/idosa)
]

VITIMA_DEFICIENCIA = [
    'DEF_TRANS',     # transtorno mental/comportamental
    'DEF_FISICA',    # deficiência física
    'DEF_MENTAL',    # deficiência intelectual
    'DEF_VISUAL',    # deficiência visual
    'DEF_AUDITI',    # deficiência auditiva
    'TRAN_MENT',     # transtorno mental específico
    'TRAN_COMP',     # transtorno comportamental
    'DEF_OUT',       # outras deficiências
]

# === ALVOS E TIPOS DE VIOLÊNCIA ===
TIPOS_VIOLENCIA = [
    'VIOL_FISIC',    # ALVO 1
    'VIOL_PSICO',    # ALVO 2
    'VIOL_SEXU',     # ALVO 3
    'VIOL_TORT',     # outros tipos (descritiva, não usar como preditor)
    'VIOL_TRAF',
    'VIOL_FINAN',
    'VIOL_NEGLI',
    'VIOL_INFAN',
    'VIOL_LEGAL',
    'VIOL_OUTR',
]

# === MEIOS DE AGRESSÃO ===
MEIOS_AGRESSAO = [
    'AG_FORCA',      # força corporal
    'AG_ENFOR',      # enforcamento
    'AG_OBJETO',     # objeto contundente
    'AG_CORTE',      # objeto perfuro-cortante
    'AG_QUENTE',     # substância/objeto quente
    'AG_ENVEN',      # envenenamento
    'AG_FOGO',       # arma de fogo
    'AG_AMEACA',     # ameaça
    'AG_OUTROS',     # outros meios
]

# === DETALHAMENTO DA VIOLÊNCIA SEXUAL ===
VIOL_SEXUAL_DETALHE = [
    'SEX_ASSEDI',    # assédio
    'SEX_ESTUPR',    # estupro
    'SEX_PORNO',     # pornografia infantil
    'SEX_EXPLO',     # exploração sexual
    'SEX_OUTRO',     # outros
]

# === RELAÇÃO COM O AGRESSOR ===
RELACAO_AGRESSOR = [
    'NUM_ENVOLV',    # número de agressores
    'REL_PAI', 'REL_MAE', 'REL_PAD', 'REL_MAD',           # familiares ascendentes
    'REL_FILHO', 'REL_IRMAO',                              # familiares descendentes/laterais
    'REL_CONJ', 'REL_EXCON', 'REL_NAMO', 'REL_EXNAM',     # parceiros
    'REL_CONHEC', 'REL_DESCO',                             # conhecido / desconhecido
    'REL_CUIDA',                                           # cuidador
    'REL_PATRAO', 'REL_TRAB', 'REL_CAT', 'REL_INST',      # contexto laboral/institucional
    'REL_POL',                                             # policial
    'REL_PROPRI',                                          # proprietário/imóvel
    'REL_OUTROS',                                          # outros
]

# === PERFIL DO AGRESSOR ===
AGRESSOR = [
    'AUTOR_SEXO',    # sexo do agressor
    'AUTOR_ALCO',    # suspeita de uso de álcool
]

# === CONTEXTO DA OCORRÊNCIA ===
CONTEXTO = [
    'LOCAL_OCOR',    # local (residência, via pública, escola, bar, etc.)
    'HORA_OCOR',     # hora aproximada
    'OUT_VEZES',     # se ocorreu antes (revitimização)
    'LES_AUTOP',     # lesão autoprovocada
]

# Lista final consolidada
COLS = (
    TEMPORAL +
    GEOGRAFICA +
    VITIMA_PERFIL +
    VITIMA_DEFICIENCIA +
    TIPOS_VIOLENCIA +
    MEIOS_AGRESSAO +
    VIOL_SEXUAL_DETALHE +
    RELACAO_AGRESSOR +
    AGRESSOR +
    CONTEXTO
)

print(f'Total de variáveis selecionadas: {len(COLS)}')
print(f'\nDistribuição por grupo:')
print(f'  Temporal:                   {len(TEMPORAL):>3}')
print(f'  Geográfica:                 {len(GEOGRAFICA):>3}')
print(f'  Vítima (perfil):            {len(VITIMA_PERFIL):>3}')
print(f'  Vítima (deficiência):       {len(VITIMA_DEFICIENCIA):>3}')
print(f'  Tipos de violência:         {len(TIPOS_VIOLENCIA):>3}')
print(f'  Meios de agressão:          {len(MEIOS_AGRESSAO):>3}')
print(f'  Violência sexual (detalhe): {len(VIOL_SEXUAL_DETALHE):>3}')
print(f'  Relação com agressor:       {len(RELACAO_AGRESSOR):>3}')
print(f'  Perfil do agressor:         {len(AGRESSOR):>3}')
print(f'  Contexto da ocorrência:     {len(CONTEXTO):>3}')

Total de variáveis selecionadas: 75

Distribuição por grupo:
  Temporal:                     4
  Geográfica:                   3
  Vítima (perfil):              9
  Vítima (deficiência):         8
  Tipos de violência:          10
  Meios de agressão:            9
  Violência sexual (detalhe):   5
  Relação com agressor:        21
  Perfil do agressor:           2
  Contexto da ocorrência:       4


## Download dos arquivos via pysus

In [5]:
from pysus import SINAN

ANOS = list(range(2018, 2025))  # 2018, 2019, ..., 2024
sinan = SINAN().load()
files = sinan.get_files(dis_code='VIOL', year=ANOS)
sinan.download(files)

paths = sorted(glob.glob('/root/pysus/VIOLBR*.parquet'))
paths = [p for p in paths if any(f'VIOLBR{a%100:02d}' in p for a in ANOS)]
print(f'Arquivos disponíveis: {len(paths)}')
for p in paths:
    print(f'  - {Path(p).name}')

VIOLBR24.parquet: 100%|██████████| 1.77M/1.77M [02:16<00:00, 12.9kB/s]

Arquivos disponíveis: 7
  - VIOLBR18.parquet
  - VIOLBR19.parquet
  - VIOLBR20.parquet
  - VIOLBR21.parquet
  - VIOLBR22.parquet
  - VIOLBR23.parquet
  - VIOLBR24.parquet


## Processamento ano a ano

Para evitar estouro de memória no Colab, processamos cada ano separadamente:
1. Lê o parquet com apenas as colunas selecionadas
2. Filtra mulheres (`CS_SEXO = F`)
3. Salva o ano processado em `partes/`

Caso a sessão caia, basta re-rodar — arquivos já processados são pulados.

In [6]:
for p in paths:
    nome = Path(p).stem
    saida = PARTES / f'{nome}.parquet'

    if saida.exists():
        print(f'{nome}: já existe, pulando')
        continue

    df_ano = pd.read_parquet(p, columns=COLS)
    n_total = len(df_ano)

    df_ano = df_ano[df_ano['CS_SEXO'] == 'F'].reset_index(drop=True)
    n_mulher = len(df_ano)

    df_ano.to_parquet(saida, index=False)
    print(f'{nome}: {n_total:>7,} -> {n_mulher:>7,} mulheres')

    del df_ano
    gc.collect()

print('\nProcessamento concluído.')

VIOLBR18: 350,354 -> 252,668 mulheres
VIOLBR19: 405,497 -> 289,742 mulheres
VIOLBR20: 326,503 -> 233,072 mulheres
VIOLBR21: 373,262 -> 268,884 mulheres
VIOLBR22: 442,680 -> 318,137 mulheres
VIOLBR23: 588,388 -> 419,967 mulheres
VIOLBR24: 616,548 -> 437,828 mulheres

Processamento concluído.


## Concatenação final

In [7]:
partes = sorted(PARTES.glob('VIOLBR*.parquet'))
print(f'Concatenando {len(partes)} partes...\n')

dfs = []
for p in partes:
    df_p = pd.read_parquet(p)
    print(f'  {p.stem}: {len(df_p):>7,}')
    dfs.append(df_p)

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f'\nBase consolidada: {df.shape[0]:,} linhas × {df.shape[1]} colunas')

Concatenando 12 partes...

  VIOLBR18: 252,668
  VIOLBR19: 289,742
  VIOLBR20: 233,072
  VIOLBR20_mulher: 233,072
  VIOLBR21: 268,884
  VIOLBR21_mulher: 268,884
  VIOLBR22: 318,137
  VIOLBR22_mulher: 318,137
  VIOLBR23: 419,967
  VIOLBR23_mulher: 419,967
  VIOLBR24: 437,828
  VIOLBR24_mulher: 437,828

Base consolidada: 3,898,186 linhas × 77 colunas


## Criação dos alvos binários e salvamento

In [8]:
# Mapeia alvos: 1=Sim, 2=Não, demais=NaN
for col in ['VIOL_FISIC', 'VIOL_PSICO', 'VIOL_SEXU']:
    novo = 'y_' + col.replace('VIOL_', '').lower()
    df[novo] = df[col].map({'1': 1, '2': 0})

print('Distribuição dos alvos:\n')
for col in ['y_fisic', 'y_psico', 'y_sexu']:
    n_pos = (df[col] == 1).sum()
    n_neg = (df[col] == 0).sum()
    n_na  = df[col].isna().sum()
    prev  = 100 * n_pos / (n_pos + n_neg) if (n_pos + n_neg) > 0 else 0
    print(f'  {col}: Sim={n_pos:>8,} | Não={n_neg:>8,} | NaN={n_na:>7,} | Prev={prev:.1f}%')

Distribuição dos alvos:

  y_fisic: Sim=1,968,423 | Não=1,880,822 | NaN= 48,941 | Prev=51.1%
  y_psico: Sim= 900,411 | Não=2,905,317 | NaN= 92,458 | Prev=23.7%
  y_sexu: Sim= 620,142 | Não=3,183,518 | NaN= 94,526 | Prev=16.3%


In [9]:
saida_final = DRIVE / 'viol_mulher.parquet'
df.to_parquet(saida_final, index=False)
print(f'\nSalvo em: {saida_final}')
print(f'Tamanho: {saida_final.stat().st_size / 1024 / 1024:.1f} MB')


Salvo em: /content/drive/MyDrive/projeto_violencia_mulher/viol_mulher.parquet
Tamanho: 71.0 MB
